In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook explores four performance characteristics of a
# continuous-time Chebyshev Type II low-pass filter:
#
# 1. Group delay
# 2. Loss characteristic
# 3. Selectivity
# 4. Spectral performance factor
#
# The sliders control:
#
#       N    : filter order
#       ωs   : stopband edge frequency
#       As   : minimum stopband attenuation
#       α, β : attenuation levels used for the spectral performance factor
#
# Chebyshev Type II filters have a monotonic passband and an equiripple
# stopband. Increasing N sharpens the transition region. The stopband zeros
# generate the characteristic equiripple loss behavior.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 7px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:760px;
    max-width:760px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Explore the group delay, loss characteristic, selectivity and spectral
performance factor of a continuous-time Chebyshev Type II low-pass filter.
<br>
<b>Interpretation:</b>
The sliders control N, ωs and the stopband attenuation As. Increasing N
sharpens the transition region, while the stopband exhibits the characteristic
equiripple behavior of the inverse-Chebyshev approximation. The attenuation
levels α and β are used only in the calculation of the spectral performance
factor.
</div>
""", layout=Layout(width='770px', max_width='770px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='250px')
style_opts = {'description_width':'75px'}

order_slider = IntSlider(min=2, max=10, step=1, value=4, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)
ws_slider = FloatSlider(min=0.5, max=3.0, step=0.1, value=1.2, description='ωs:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)
as_slider = FloatSlider(min=20.0, max=80.0, step=1.0, value=50.0, description='As (dB):', continuous_update=True, readout=True, readout_format='.0f', style=style_opts, layout=slider_layout)
alpha_slider = FloatSlider(min=1.0, max=10.0, step=0.5, value=3.0, description='α (dB):', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)
beta_slider = FloatSlider(min=20.0, max=80.0, step=1.0, value=40.0, description='β (dB):', continuous_update=True, readout=True, readout_format='.0f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='280px', max_width='280px'))

# ==============================================================================
# FIGURE 1: GROUP DELAY
# ==============================================================================

fig_gd, ax_gd = plt.subplots(figsize=(5.2, 3.15))

gd_line, = ax_gd.plot([], [], 'r-', linewidth=2.0, label='τg(ω)')
ws_gd_line = ax_gd.axvline(1.2, color='black', linestyle=':', linewidth=1.2, label='ωs')

ax_gd.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)
ax_gd.set_ylabel('Group Delay τg(ω) (s)', fontsize=10)
ax_gd.set_title('Group Delay', fontsize=12, fontweight='bold', pad=5)
ax_gd.tick_params(axis='both', labelsize=9)
ax_gd.grid(True, linestyle=':', alpha=0.5)
ax_gd.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=8)

# Fixed axes
ax_gd.set_xlim(0.0, 10.0)
ax_gd.set_ylim(0.0, 80.0)
ax_gd.set_yticks(np.arange(0.0, 81.0, 10.0))

fig_gd.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.86)
fig_gd.canvas.header_visible = False
fig_gd.canvas.toolbar_visible = False
fig_gd.canvas.resizable = False
fig_gd.canvas.layout.width = '520px'
fig_gd.canvas.layout.height = '320px'

# ==============================================================================
# FIGURE 2: LOSS CHARACTERISTIC
# ==============================================================================

fig_loss, ax_loss = plt.subplots(figsize=(5.2, 3.15))

loss_line, = ax_loss.plot([], [], 'r-', linewidth=2.0, label='A(ω)')
ws_loss_line = ax_loss.axvline(1.2, color='black', linestyle=':', linewidth=1.2, label='ωs')
as_loss_line = ax_loss.axhline(50.0, color='gray', linestyle='--', linewidth=1.0, label='As')

ax_loss.set_xscale('log')
ax_loss.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)
ax_loss.set_ylabel('Loss A(ω) (dB)', fontsize=10)
ax_loss.set_title('Loss Characteristic', fontsize=12, fontweight='bold', pad=5)
ax_loss.tick_params(axis='both', labelsize=9)
ax_loss.grid(True, which='both', linestyle=':', alpha=0.5)
ax_loss.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=3, fontsize=8)

ax_loss.set_xlim(1e-4, 1e2)
ax_loss.set_ylim(0.0, 110.0)
ax_loss.set_yticks(np.arange(0.0, 101.0, 10.0))

fig_loss.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.86)
fig_loss.canvas.header_visible = False
fig_loss.canvas.toolbar_visible = False
fig_loss.canvas.resizable = False
fig_loss.canvas.layout.width = '520px'
fig_loss.canvas.layout.height = '320px'

# ==============================================================================
# FIGURE 3: SELECTIVITY
# ==============================================================================

fig_sel, ax_sel = plt.subplots(figsize=(5.2, 3.15))

sel_line, = ax_sel.plot([], [], 'r-', linewidth=2.0, label='Fs(N)')
sel_point, = ax_sel.plot([], [], 'ro', markersize=6, label='Current N')

ax_sel.set_xlabel('Filter Order N', fontsize=10)
ax_sel.set_ylabel('Selectivity Fs', fontsize=10)
ax_sel.set_title('Selectivity', fontsize=12, fontweight='bold', pad=5)
ax_sel.tick_params(axis='both', labelsize=9)
ax_sel.grid(True, linestyle=':', alpha=0.5)
ax_sel.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=8)

ax_sel.set_xlim(1.0, 10.0)
ax_sel.set_xticks(np.arange(1, 11, 1))
ax_sel.set_ylim(0.0, 60.0)
ax_sel.set_yticks(np.arange(0.0, 61.0, 10.0))

fig_sel.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.86)
fig_sel.canvas.header_visible = False
fig_sel.canvas.toolbar_visible = False
fig_sel.canvas.resizable = False
fig_sel.canvas.layout.width = '520px'
fig_sel.canvas.layout.height = '320px'

# ==============================================================================
# FIGURE 4: SPECTRAL PERFORMANCE FACTOR
# ==============================================================================

fig_spf, ax_spf = plt.subplots(figsize=(5.2, 3.15))

spf_line, = ax_spf.plot([], [], 'r-', linewidth=2.0, label='Sαβ(N)')
spf_point, = ax_spf.plot([], [], 'ro', markersize=6, label='Current N')
ideal_spf_line = ax_spf.axhline(1.0, color='gray', linestyle='--', linewidth=1.0, label='Ideal = 1')

ax_spf.set_xlabel('Filter Order N', fontsize=10)
ax_spf.set_ylabel('Spectral Factor Sαβ', fontsize=10)
ax_spf.set_title('Spectral Performance Factor', fontsize=12, fontweight='bold', pad=5)
ax_spf.tick_params(axis='both', labelsize=9)
ax_spf.grid(True, linestyle=':', alpha=0.5)
ax_spf.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=3, fontsize=8)

ax_spf.set_xlim(1.0, 10.0)
ax_spf.set_xticks(np.arange(1, 11, 1))
ax_spf.set_ylim(0.0, 20.0)
ax_spf.set_yticks(np.arange(0.0, 21.0, 5.0))

fig_spf.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.86)
fig_spf.canvas.header_visible = False
fig_spf.canvas.toolbar_visible = False
fig_spf.canvas.resizable = False
fig_spf.canvas.layout.width = '520px'
fig_spf.canvas.layout.height = '320px'

# ==============================================================================
# AUXILIARY FUNCTIONS
# ==============================================================================

def cheby2_system(N, As, ws):
    z, p, k = signal.cheby2(N, As, ws, btype='low', analog=True, output='zpk')
    b, a = signal.zpk2tf(z, p, k)
    return b, a

def attenuation_frequency(b, a, level_db, w_min=1e-5, w_max=1e3):
    w = np.logspace(np.log10(w_min), np.log10(w_max), 30000)
    _, H = signal.freqs(b, a, worN=w)
    A = -20.0 * np.log10(np.maximum(np.abs(H), 1e-15))
    indices = np.where(A >= level_db)[0]
    if len(indices) == 0:
        return np.nan
    return w[indices[0]]

def performance_quantities(N, As, ws, alpha_db, beta_db):
    b, a = cheby2_system(N, As, ws)
    wa = attenuation_frequency(b, a, alpha_db)
    wb = attenuation_frequency(b, a, beta_db)
    if np.isnan(wa) or np.isnan(wb) or wa <= 0.0 or wb <= wa:
        return np.nan, np.nan, np.nan, np.nan
    bw = wb - wa
    bw_percent = 100.0 * bw / wa
    Fs = wa / bw
    Sab = wb / wa
    return Fs, bw, bw_percent, Sab

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_chebyshev2(change=None):

    N = order_slider.value
    ws = ws_slider.value
    As = as_slider.value
    alpha_db = alpha_slider.value
    beta_db = beta_slider.value

    # --------------------------------------------------------------------------
    # Construct Chebyshev Type II analog low-pass filter
    # --------------------------------------------------------------------------

    b, a = cheby2_system(N, As, ws)

    # --------------------------------------------------------------------------
    # Frequency response for group delay
    # --------------------------------------------------------------------------

    omega_gd = np.linspace(0.001, 10.0, 5000)
    _, H_gd = signal.freqs(b, a, worN=omega_gd)

    phase = np.unwrap(np.angle(H_gd))
    group_delay = -np.gradient(phase, omega_gd)

    # Remove tiny negative numerical artifacts
    group_delay = np.where(group_delay >= 0.0, group_delay, np.nan)

    # --------------------------------------------------------------------------
    # Frequency response for loss characteristic
    # --------------------------------------------------------------------------

    omega_loss = np.logspace(-4, 2, 6000)
    _, H_loss = signal.freqs(b, a, worN=omega_loss)

    magnitude = np.maximum(np.abs(H_loss), 1e-15)
    loss_db = -20.0 * np.log10(magnitude)

    # --------------------------------------------------------------------------
    # Performance curves versus filter order
    # --------------------------------------------------------------------------

    orders = np.arange(1, 11)

    selectivity_values = []
    spectral_values = []

    for n in orders:
        Fs_n, bw_n, bwp_n, Sab_n = performance_quantities(n, As, ws, alpha_db, beta_db)
        selectivity_values.append(Fs_n)
        spectral_values.append(Sab_n)

    selectivity_values = np.asarray(selectivity_values, dtype=float)
    spectral_values = np.asarray(spectral_values, dtype=float)

    Fs, BW, BWP, Sab = performance_quantities(N, As, ws, alpha_db, beta_db)

    # --------------------------------------------------------------------------
    # Group delay update
    # --------------------------------------------------------------------------

    gd_line.set_data(omega_gd, group_delay)
    ws_gd_line.set_xdata([ws, ws])

    ax_gd.set_xlim(0.0, 10.0)

    # IMPORTANT: fixed vertical scale requested
    ax_gd.set_ylim(0.0, 80.0)
    ax_gd.set_yticks(np.arange(0.0, 81.0, 10.0))

    # --------------------------------------------------------------------------
    # Loss characteristic update
    # --------------------------------------------------------------------------

    loss_line.set_data(omega_loss, loss_db)
    ws_loss_line.set_xdata([ws, ws])
    as_loss_line.set_ydata([As, As])

    ax_loss.set_xlim(1e-4, 1e2)
    ax_loss.set_ylim(0.0, 110.0)

    # --------------------------------------------------------------------------
    # Selectivity update
    # --------------------------------------------------------------------------

    sel_line.set_data(orders, selectivity_values)

    if np.isfinite(Fs):
        sel_point.set_data([N], [Fs])
    else:
        sel_point.set_data([], [])

    ax_sel.set_xlim(1.0, 10.0)
    ax_sel.set_ylim(0.0, 60.0)

    # --------------------------------------------------------------------------
    # Spectral performance factor update
    # --------------------------------------------------------------------------

    spf_line.set_data(orders, spectral_values)

    if np.isfinite(Sab):
        spf_point.set_data([N], [Sab])
    else:
        spf_point.set_data([], [])

    ax_spf.set_xlim(1.0, 10.0)
    ax_spf.set_ylim(0.0, 20.0)

    # --------------------------------------------------------------------------
    # Information panel
    # --------------------------------------------------------------------------

    if np.isfinite(Fs):
        Fs_text = f'{Fs:.6f}'
    else:
        Fs_text = 'undefined'

    if np.isfinite(BW):
        BW_text = f'{BW:.6f} rad/s'
    else:
        BW_text = 'undefined'

    if np.isfinite(BWP):
        BWP_text = f'{BWP:.6f}%'
    else:
        BWP_text = 'undefined'

    if np.isfinite(Sab):
        Sab_text = f'{Sab:.6f}'
    else:
        Sab_text = 'undefined'

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 9px;
        margin-top:9px;
        font-size:12px;
        line-height:1.55;
        background:white;
        width:275px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Chebyshev Type II low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Stopband edge:</b>
        <span style="color:#0066cc;">ωs = {ws:.2f} rad/s</span>
    </div>

    <div>
        <b>Stopband attenuation:</b>
        <span style="color:#0066cc;">As = {As:.1f} dB</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Performance quantities:</b>
    </div>

    <div>
        <b>Selectivity Fs:</b>
        <span style="color:#0066cc;">{Fs_text}</span>
    </div>

    <div>
        <b>Attenuation levels:</b>
        <span style="color:#0066cc;">α = {alpha_db:.1f} dB, β = {beta_db:.1f} dB</span>
    </div>

    <div>
        <b>BWβ−α:</b>
        <span style="color:#0066cc;">{BW_text}</span>
    </div>

    <div>
        <b>BWβ−α (%):</b>
        <span style="color:#0066cc;">{BWP_text}</span>
    </div>

    <div>
        <b>Spectral factor Sαβ:</b>
        <span style="color:#0066cc;">{Sab_text}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        Increasing N sharpens the transition region. Unlike Chebyshev Type I,
        the passband is monotonic while the stopband is equiripple. The
        transmission zeros are responsible for the characteristic stopband
        peaks in the loss response. The group delay is frequency dependent
        and varies most strongly around the transition region.
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # REDRAW EXISTING FIGURES ONLY
    # --------------------------------------------------------------------------

    fig_gd.canvas.draw_idle()
    fig_loss.canvas.draw_idle()
    fig_sel.canvas.draw_idle()
    fig_spf.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_chebyshev2, names='value')
ws_slider.observe(update_chebyshev2, names='value')
as_slider.observe(update_chebyshev2, names='value')
alpha_slider.observe(update_chebyshev2, names='value')
beta_slider.observe(update_chebyshev2, names='value')

# ==============================================================================
# LAYOUT: 2 x 2 FIGURE GRID
# ==============================================================================

controls = VBox([parameter_title, order_slider, ws_slider, as_slider, alpha_slider, beta_slider, info_html], layout=Layout(width='290px', min_width='290px', max_width='290px', flex='0 0 290px', align_items='flex-start'))

top_row = HBox([fig_gd.canvas, fig_loss.canvas], layout=Layout(width='1050px', align_items='flex-start', justify_content='flex-start'))

bottom_row = HBox([fig_sel.canvas, fig_spf.canvas], layout=Layout(width='1050px', align_items='flex-start', justify_content='flex-start'))

plot_grid = VBox([top_row, bottom_row], layout=Layout(width='1050px', align_items='flex-start', justify_content='flex-start'))

main_layout = HBox([controls, plot_grid], layout=Layout(width='1340px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE DATA
# ==============================================================================

update_chebyshev2()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_layout)